# Spring Boot 4.x Migration Assistant

**Iterative prompting applied to a real multi-library upgrade scenario.**

Pattern: **discovery-driven iteration** — each Claude response unblocks the next layer of failures.  
Unlike prompt refinement, here the *problem* changes each turn, not just the phrasing.



In [14]:
from anthropic import Anthropic
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
SYSTEM_PROMPT = "You are a senior Java and Spring Boot engineer."
MODEL = "claude-sonnet-4-6"
MAX_TOKENS = 2048
response = anthropic_client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.content[0].text)


Hello there! I hope you're having a wonderful day! 😊


In [ ]:
def chat(messages, user_message):
    messages.append({"role": "user", "content": user_message})
    response = anthropic_client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM_PROMPT, messages=messages
    )
    print(response.content[0].text)
    messages.append({"role": "assistant", "content": response.content[0].text})
    return response


## Library A — `@MockBean` Removal

**Library**: `app-test-support:2.1.4` — provides `AbstractServiceTest`, a base class used by 47 test classes across the platform.

**Breaking change in Spring Boot 4.0**: `@MockBean` and `@SpyBean` were fully removed from  
`org.springframework.boot.test.mock.mockito`. Replacement: `@MockitoBean` / `@MockitoSpyBean`  
from `org.springframework.test.context.bean.override.mockito`.

**The loop below**:
1. Ask Claude to upgrade `build.gradle` to Spring Boot 4.0.6
2. Run `./gradlew clean build` → compilation fails on `@MockBean` *(simulated)*
3. Send the error back to Claude and apply the fix
4. Final check before cutting a new library release


In [ ]:
# Turn 1 — upgrade the Gradle build file
messages = []

turn1 = """We are already on Spring Boot 3.2.5 (not upgrading from 2.x),
so javax→jakarta and spring.factories migrations are already complete.
Focus on breaking changes specific to the 3.x → 4.0 upgrade.

Spring Boot 4.0.6 is the current production release as of May 2026.
I'm upgrading the Mark orchestator platform from Spring Boot 3.2.5 to 4.0.6.
Here is the current build.gradle for our internal library fh-test-support.
Please update the Spring Boot plugin version, the dependency-management plugin,
and any related configuration to target Spring Boot 4.0.6.

plugins {
    id 'java-library'
    id 'org.springframework.boot' version '3.2.5' apply false
    id 'io.spring.dependency-management' version '1.1.4'
}

group = 'com.fh.platform'
version = '2.1.4'
sourceCompatibility = '17'

dependencyManagement {
    imports {
        mavenBom "org.springframework.boot:spring-boot-dependencies:3.2.5"
    }
}

dependencies {
    api 'org.springframework.boot:spring-boot-starter'
    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}

What should change and why?"""

r1 = chat(messages, turn1)
 

## Changes Required and Rationale

### Updated `build.gradle`

```groovy
plugins {
    id 'java-library'
    id 'org.springframework.boot' version '4.0.6' apply false
    id 'io.spring.dependency-management' version '1.1.7'
}

group = 'com.fh.platform'
version = '2.1.4'
sourceCompatibility = '21'

dependencyManagement {
    imports {
        mavenBom "org.springframework.boot:spring-boot-dependencies:4.0.6"
    }
}

dependencies {
    api 'org.springframework.boot:spring-boot-starter'
    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}
```

---

### What Changed and Why

#### 1. Spring Boot plugin version `3.2.5` → `4.0.6`
The plugin version must match the BOM version exactly. A mismatch causes the
plugin's built-in dependency resolution hints and repackaging logic to target
the wrong managed versions, silently pulling in incompatible transitive
dependencies.

#### 2. `io.spring.dependency-management` plugin `1.1.4` → `1.1.7`
Spring Boot 4.x ships BOMs that use

### Simulated: `./gradlew clean build` after applying the update

```
> Task :fh-test-support:compileTestJava FAILED

FAILURE: Build failed with an exception.

* What went wrong:
Execution failed for task ':fh-test-support:compileTestJava'.
> Compilation failed; see the compiler error output for details.

src/test/java/com/fh/service/UserServiceTest.java:8: error: package org.springframework.boot.test.mock.mockito does not exist
import org.springframework.boot.test.mock.mockito.MockBean;
                                          ^
src/test/java/com/fh/service/UserServiceTest.java:19: error: cannot find symbol
    @MockBean
     ^
  symbol:   class MockBean

BUILD FAILED in 4s
```

Gradle resolved correctly — BOM update worked. But `@MockBean` no longer exists in Spring Boot 4.0.


In [ ]:
# Turn 2 — send the error, ask for the fix (messages list carries context from Turn 1)
turn2 = """The build.gradle changes resolved correctly. But ./gradlew clean build failed during test compilation:

    src/test/java/com/fh/service/UserServiceTest.java:8:
    error: package org.springframework.boot.test.mock.mockito does not exist
    import org.springframework.boot.test.mock.mockito.MockBean;

    src/test/java/com/fh/service/UserServiceTest.java:19: error: cannot find symbol
        @MockBean

Here is one of the 47 failing test classes:

    @SpringBootTest
    class UserServiceTest {

        @Autowired
        private UserService userService;

        @MockBean
        private UserRepository userRepository;

        @MockBean
        @Primary
        private CacheManager cacheManager;

        @Test
        void shouldReturnUser() { ... }
    }

What is the correct replacement for @MockBean in Spring Boot 4.0?
Also: one mock uses @Primary — does that interact differently with the replacement annotation?"""

r2 = chat(messages, turn2)


## Root Cause

`@MockBean` (and `@SpyBean`) were removed from `spring-boot-test` in Spring Boot 4.0. The entire `org.springframework.boot.test.mock.mockito` package is gone. This functionality was migrated into Spring Framework itself and now lives in `spring-test` as a first-class feature, no longer requiring the Boot wrapper.

---

## The Replacement

| Removed (Boot 3.x) | Replacement (Boot 4.0 / Spring Framework 7.x) |
|---|---|
| `org.springframework.boot.test.mock.mockito.MockBean` | `org.springframework.test.context.bean.override.mockito.MockitoBean` |
| `org.springframework.boot.test.mock.mockito.SpyBean` | `org.springframework.test.context.bean.override.mockito.MockitoSpyBean` |

These are in `spring-test`, which is already on your test classpath transitively through `spring-boot-starter-test`. No new dependency is needed.

---

## Corrected Test Class

```java
import org.springframework.test.context.bean.override.mockito.MockitoBean;  // replaces @MockBean
// @Primary handlin